In [1]:
import hashlib
import json
import warnings
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
import torch.nn.functional as F
from scipy import stats
from scipy.stats import ttest_rel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("default")

# Fixed seeds: random item selection and bootstrap uncertainty are reproducible.
RANDOM_SEED = 42
BOOTSTRAP_SEED = 2026
N_BOOTSTRAP = 10_000

# ============================================================================
# Dataset loading and reproducible item sampling
# ============================================================================

def load_prolific_dataset(
    filepath: str,
    max_samples: int = 393,
    seed: int = RANDOM_SEED,
) -> List[Dict]:
    """Load a one-answer-per-question JSONL and sample unique questions.

    Expected schema per line: {"question": "...", "answer": "..."}.
    Sampling is simple random sampling without replacement, with a fixed seed.
    The reduced input must contain one record per distinct question.
    """
    path = Path(filepath)
    print(f"\nLoading reduced Social Reasoning dataset from {path}...")
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {path}. Set _prolific_file to the path of "
            "the reduced JSONL with 'question' and 'answer' fields (e.g. "
            "most_preferred.jsonl). The original pairwise file with 'chosen' "
            "and 'rejected' is not the expected input for this experiment."
        )
    if max_samples is None or max_samples < 1:
        raise ValueError(f"max_samples must be a positive integer; got {max_samples!r}")

    examples = []
    seen_questions = set()
    with path.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {line_num} of {path}: {exc}") from exc

            question = item.get("question", "")
            answer = item.get("answer", "")
            if not isinstance(question, str) or not isinstance(answer, str):
                raise ValueError(
                    f"Line {line_num} must contain string 'question' and 'answer' fields."
                )
            question = question.strip()
            answer = answer.strip()
            if not question or not answer:
                raise ValueError(f"Line {line_num} has an empty question or answer.")
            if question in seen_questions:
                raise ValueError(
                    f"Duplicate question at line {line_num}. The input must have "
                    "exactly one record per distinct question."
                )
            seen_questions.add(question)
            examples.append({
                "prompt": question,
                "response": answer,
                "source": "ProlificAI",
                # Stable identifier used to verify exact alignment across models.
                "question_id": hashlib.sha256(question.encode("utf-8")).hexdigest(),
            })

    if len(examples) < max_samples:
        raise ValueError(
            f"Requested {max_samples} items, but only {len(examples)} unique "
            f"question-answer records were found in {path}."
        )

    rng = np.random.default_rng(seed)
    selected_indices = rng.choice(len(examples), size=max_samples, replace=False)
    selected = [examples[int(i)] for i in selected_indices]

    print(f"  Available unique questions: {len(examples)}")
    print(f"  Randomly selected: {len(selected)} (seed={seed}, no replacement)")
    return selected


In [2]:
# ============================================================================
# Full-sequence loss calculation and question-level inference
# ============================================================================

def get_context_limit(model, tokenizer) -> int:
    """Return a conservative usable context limit without tokenizer sentinels."""
    config = model.config
    for attr in ("max_position_embeddings", "n_positions", "max_sequence_length", "seq_length"):
        value = getattr(config, attr, None)
        if isinstance(value, int) and 2 <= value < 10_000_000:
            return value
    value = getattr(tokenizer, "model_max_length", None)
    if isinstance(value, int) and 2 <= value < 10_000_000:
        return value
    raise ValueError(
        "Could not infer the model's context limit from its configuration or tokenizer. "
        "Set a valid context limit explicitly rather than truncating examples silently."
    )


def compute_loss(model, tokenizer, prompt: str, response: str):
    """Return mean full-sequence next-token NLL, token count, and windowing flag.

    The prompt and selected answer are tokenized together. No tokens are silently
    truncated. If the sequence exceeds the model's usable context limit, sliding
    context windows score every next-token target exactly once; each token is
    conditioned on the preceding context available within that window.
    """
    model.eval()
    device = model.device
    full_text = prompt + " " + response
    encoded = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=False,
        add_special_tokens=True,
    )
    input_ids = encoded["input_ids"].to(device)
    sequence_length = int(input_ids.shape[1])
    if sequence_length < 2:
        raise ValueError("Tokenized prompt-plus-answer must contain at least two tokens.")

    context_limit = get_context_limit(model, tokenizer)
    if sequence_length <= context_limit:
        with torch.no_grad():
            outputs = model(input_ids=input_ids, labels=input_ids, use_cache=False)
            loss = float(outputs.loss.detach().float().cpu().item())
        return loss, sequence_length, False

    # Long examples: score all target tokens once, with bounded left context.
    # Every target position 1..L-1 is covered; no prefix or suffix is dropped.
    stride = max(1, context_limit // 2)
    nll_sum = 0.0
    n_scored_tokens = 0
    sequence_len = sequence_length
    with torch.no_grad():
        for target_start in range(1, sequence_len, stride):
            target_end = min(target_start + stride, sequence_len)
            context_start = max(0, target_end - context_limit)
            window_ids = input_ids[:, context_start:target_end]
            outputs = model(input_ids=window_ids, use_cache=False)
            logits = outputs.logits

            # Token at absolute position t is predicted by the logit at t-1.
            local_start = target_start - context_start
            local_end = target_end - context_start
            prediction_logits = logits[:, local_start - 1:local_end - 1, :].float()
            target_ids = input_ids[:, target_start:target_end]
            chunk_nll = F.cross_entropy(
                prediction_logits.reshape(-1, prediction_logits.shape[-1]),
                target_ids.reshape(-1),
                reduction="sum",
            )
            nll_sum += float(chunk_nll.detach().cpu().item())
            n_scored_tokens += int(target_ids.numel())
            del outputs, logits, prediction_logits, chunk_nll

    expected_targets = sequence_length - 1
    if n_scored_tokens != expected_targets:
        raise RuntimeError(
            f"Internal token-accounting error: scored {n_scored_tokens} targets; "
            f"expected {expected_targets}."
        )
    return nll_sum / n_scored_tokens, sequence_length, True


def _bootstrap_mean_ci(values, seed: int = BOOTSTRAP_SEED, n_bootstrap: int = N_BOOTSTRAP):
    """Percentile 95% bootstrap CI for the mean over sampled question items."""
    values = np.asarray(values, dtype=float)
    if values.size < 2:
        raise ValueError("At least two item-level observations are needed for a CI.")
    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_bootstrap, dtype=float)
    # Batching limits memory use while keeping deterministic results.
    batch_size = 500
    start = 0
    while start < n_bootstrap:
        size = min(batch_size, n_bootstrap - start)
        idx = rng.integers(0, values.size, size=(size, values.size))
        boot_means[start:start + size] = values[idx].mean(axis=1)
        start += size
    return np.percentile(boot_means, [2.5, 97.5]).astype(float).tolist()


def compute_statistics_on_examples(
    model,
    tokenizer,
    examples: List[Dict],
    batch_size: int = 4,
) -> Dict:
    """Score each unique question once and summarize full-sequence loss."""
    all_losses = []
    question_ids = []
    sequence_lengths = []
    windowed_count = 0

    for example in tqdm(examples, desc="Computing full-sequence losses"):
        loss, sequence_length, used_windows = compute_loss(
            model, tokenizer, example["prompt"], example["response"]
        )
        all_losses.append(loss)
        question_ids.append(example["question_id"])
        sequence_lengths.append(sequence_length)
        windowed_count += int(used_windows)

    losses = np.asarray(all_losses, dtype=float)
    if len(losses) != len(examples) or len(question_ids) != len(examples):
        raise RuntimeError("The number of scores does not match the number of selected items.")
    if not np.all(np.isfinite(losses)):
        raise ValueError("Non-finite loss found; refusing to report incomplete statistics.")

    mean_loss = float(np.mean(losses))
    std_loss = float(np.std(losses, ddof=1))
    bootstrap_ci = _bootstrap_mean_ci(losses)
    return {
        "question_ids": question_ids,
        "losses": losses.tolist(),
        "n_examples": int(len(losses)),
        "mean_loss": mean_loss,
        "std_loss": std_loss,
        "ci_95_lower": float(bootstrap_ci[0]),
        "ci_95_upper": float(bootstrap_ci[1]),
        # Diagnostics for reproducibility; full-sequence loss remains the sole score.
        "n_windowed_examples": int(windowed_count),
        "max_tokenized_sequence_length": int(max(sequence_lengths)),
        "context_limit_tokens": int(get_context_limit(model, tokenizer)),
    }


def _holm_adjust(p_values):
    """Holm step-down family-wise-error adjustment, preserving input order."""
    p_values = np.asarray(p_values, dtype=float)
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        value = min(1.0, (m - rank) * p_values[idx])
        running_max = max(running_max, value)
        adjusted[idx] = running_max
    return adjusted.tolist()


def compute_pairwise_statistics(
    results_by_model: Dict[str, Dict],
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = BOOTSTRAP_SEED,
) -> Dict:
    """Paired question-level tests and bootstrap CIs, with Holm correction.

    Each model is evaluated on the identical selected questions. The bootstrap
    resamples question IDs, carrying both models' scores together (equivalently,
    resampling the vector of within-question paired differences).
    """
    comparisons = {}
    raw_p_values = []
    model_names = list(results_by_model.keys())
    rng = np.random.default_rng(seed)

    for i, model1 in enumerate(model_names):
        for model2 in model_names[i + 1:]:
            result1 = results_by_model[model1]
            result2 = results_by_model[model2]
            ids1 = result1.get("question_ids", [])
            ids2 = result2.get("question_ids", [])
            if ids1 != ids2:
                raise ValueError(
                    f"Cannot compare {model1} and {model2}: question IDs or ordering differ."
                )
            losses1 = np.asarray(result1["losses"], dtype=float)
            losses2 = np.asarray(result2["losses"], dtype=float)
            if losses1.shape != losses2.shape or losses1.size < 2:
                raise ValueError(f"Unmatched or insufficient paired losses for {model1} vs {model2}.")

            # Positive difference means model 1 has higher (worse) loss.
            differences = losses1 - losses2
            t_stat, p_value = ttest_rel(losses1, losses2, nan_policy="raise")
            mean_diff = float(np.mean(differences))
            sd_diff = float(np.std(differences, ddof=1))
            cohens_dz = float(mean_diff / sd_diff) if sd_diff > 0 else 0.0

            boot_means = np.empty(n_bootstrap, dtype=float)
            batch_size = 500
            start = 0
            while start < n_bootstrap:
                size = min(batch_size, n_bootstrap - start)
                idx = rng.integers(0, differences.size, size=(size, differences.size))
                boot_means[start:start + size] = differences[idx].mean(axis=1)
                start += size
            bootstrap_ci = np.percentile(boot_means, [2.5, 97.5]).astype(float).tolist()

            if abs(cohens_dz) < 0.2:
                effect_interpretation = "negligible"
            elif abs(cohens_dz) < 0.5:
                effect_interpretation = "small"
            elif abs(cohens_dz) < 0.8:
                effect_interpretation = "medium"
            else:
                effect_interpretation = "large"

            mean1 = float(np.mean(losses1))
            mean2 = float(np.mean(losses2))
            if mean1 < mean2:
                winner = model1.split("/")[-1]
                advantage_pct = ((mean2 - mean1) / abs(mean2) * 100.0) if mean2 != 0 else 0.0
            elif mean2 < mean1:
                winner = model2.split("/")[-1]
                advantage_pct = ((mean1 - mean2) / abs(mean1) * 100.0) if mean1 != 0 else 0.0
            else:
                winner = "tie"
                advantage_pct = 0.0

            comp_key = f"{model1.split('/')[-1]}_vs_{model2.split('/')[-1]}"
            comparisons[comp_key] = {
                "model1": model1.split("/")[-1],
                "model2": model2.split("/")[-1],
                "mean_loss_diff": mean_diff,
                "t_statistic": float(t_stat),
                "p_value_raw": float(p_value),
                "cohens_dz_paired": cohens_dz,
                "effect_size": effect_interpretation,
                "bootstrap_ci_95": [float(bootstrap_ci[0]), float(bootstrap_ci[1])],
                "winner_lower_loss": winner,
                "advantage_percent": float(advantage_pct),
                "test_type": "paired_ttest_by_question",
            }
            raw_p_values.append(float(p_value))

    adjusted_p_values = _holm_adjust(raw_p_values)
    for comp_data, p_adj in zip(comparisons.values(), adjusted_p_values):
        comp_data["p_value_holm"] = float(p_adj)
        comp_data["significant_after_holm_05"] = bool(p_adj < 0.05)

    return comparisons


# ============================================================================
# Power-analysis reference calculation (target N remains fixed at 393)
# ============================================================================

def compute_required_sample_size(
    effect_size: float = 0.2,
    alpha: float = 0.05,
    power: float = 0.80,
) -> int:
    """Original normal-approximation reference calculation for item-count planning.

    This formula is the two-independent-group approximation. The experiment keeps
    its resulting target N=393 as specified; model comparisons themselves are
    paired because every model is scored on the same selected questions.
    """
    from scipy.stats import norm
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    n = ((z_alpha + z_beta) ** 2) * 2 / (effect_size ** 2)
    return int(np.ceil(n))


# ============================================================================
# Visualization (loss and paired effect sizes; no p-value heatmap)
# ============================================================================

def generate_visualizations(results: Dict, output_dir: str = "."):
    """Create full-sequence loss CIs and paired-effect-size heatmaps."""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
    except ImportError:
        print("\nMatplotlib/seaborn not installed; skipping visualizations.")
        return

    output_dir = Path(output_dir)
    models = list(results["model_results"].keys())
    model_names = [m.split("/")[-1] for m in models]
    n_models = len(models)
    if n_models == 0:
        return

    means = [results["model_results"][m]["mean_loss"] for m in models]
    ci_lowers = [results["model_results"][m]["ci_95_lower"] for m in models]
    ci_uppers = [results["model_results"][m]["ci_95_upper"] for m in models]
    errors_lower = [max(0.0, means[i] - ci_lowers[i]) for i in range(n_models)]
    errors_upper = [max(0.0, ci_uppers[i] - means[i]) for i in range(n_models)]

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(n_models)
    ax.bar(x, means, yerr=[errors_lower, errors_upper], capsize=5, alpha=0.7)
    ax.set_xlabel("Model")
    ax.set_ylabel("Mean full-sequence loss (lower is better)")
    ax.set_title("Social Reasoning Full-Sequence Loss (95% bootstrap CI)")
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=45, ha="right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    loss_path = output_dir / "loss_comparison_with_ci.png"
    plt.savefig(loss_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {loss_path}")
    plt.close()

    effect_matrix = np.zeros((n_models, n_models), dtype=float)
    for comp_data in results.get("pairwise_statistics", {}).values():
        m1_name, m2_name = comp_data["model1"], comp_data["model2"]
        i, j = model_names.index(m1_name), model_names.index(m2_name)
        effect = comp_data["cohens_dz_paired"]
        effect_matrix[i, j] = effect
        effect_matrix[j, i] = -effect

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        effect_matrix, annot=True, fmt=".3f", cmap="RdBu_r",
        xticklabels=model_names, yticklabels=model_names, center=0,
        cbar_kws={"label": "Paired Cohen's dz"},
    )
    plt.title("Paired Effect Sizes (positive = row model has higher/worse loss)")
    plt.tight_layout()
    effect_path = output_dir / "effect_size_heatmap.png"
    plt.savefig(effect_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {effect_path}")
    plt.close()


# ============================================================================
# Model evaluation
# ============================================================================

def evaluate_model(model_path: str, examples: List[Dict], batch_size: int = 4) -> Dict:
    """Load a model and evaluate the fixed item sample using full-sequence loss."""
    print(f"\n{'=' * 80}\nEvaluating: {model_path}\n{'=' * 80}")
    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model.eval()

    results = compute_statistics_on_examples(model, tokenizer, examples, batch_size)
    print(f"\nResults for {model_path}:")
    print(f"  Items: {results['n_examples']}")
    print(f"  Mean full-sequence loss: {results['mean_loss']:.4f}")
    print(f"  SD across items: {results['std_loss']:.4f}")
    print(f"  95% bootstrap CI: [{results['ci_95_lower']:.4f}, {results['ci_95_upper']:.4f}]")
    print(
        f"  Long examples scored with sliding windows: {results['n_windowed_examples']} "
        f"(max tokenized length={results['max_tokenized_sequence_length']}, "
        f"context limit={results['context_limit_tokens']})"
    )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results


In [4]:
# ============================================================================
# Main
# ============================================================================

_visualize = True
_models = [
    "Qwen3-8B",
    "Qwen3-8B-Base",
    "llama-3.1-8b-base",
    "llama-3.1-8B-instruct",
]
_output = "social_reasoning_baseline_revised.json"
_max_samples = None  # Use the fixed target from the stated power calculation (393).
_batch_size = 4
_prolific_file = "most_preferred.jsonl"  # Reduced schema: one {question, answer} per line.
_sampling_seed = RANDOM_SEED
_skip_prolific = False

print("=" * 80)
print("SOCIAL REASONING EVALUATION")
print("=" * 80)

required_n = compute_required_sample_size(effect_size=0.2, power=0.80)
print("\nPower-analysis reference:")
print("  Effect size parameter: d=0.20 (small standardized effect)")
print("  Nominal power: 80%; two-sided alpha: 0.05")
print(f"  Fixed target sample: {required_n} unique questions")
print("  Model comparisons are paired by question; the original sample-size formula is retained as the planning reference.")

if _max_samples is None:
    _max_samples = required_n
else:
    print(f"  Using user-specified item count: {_max_samples}")

examples = []
if not _skip_prolific:
    examples = load_prolific_dataset(
        _prolific_file,
        max_samples=_max_samples,
        seed=_sampling_seed,
    )
if not examples:
    raise RuntimeError("No examples loaded; evaluation stopped.")

print(f"\nTotal evaluated items: {len(examples)}")
print(f"Unique question IDs: {len({e['question_id'] for e in examples})}")

all_results = {
    "metadata": {
        "dataset_file": _prolific_file,
        "n_examples_total": len(examples),
        "sampling_method": "simple_random_sampling_without_replacement",
        "sampling_seed": _sampling_seed,
        "power_analysis": {
            "target_power": 0.80,
            "effect_size_d": 0.2,
            "alpha_two_sided": 0.05,
            "required_n": required_n,
            "note": "N is retained from the original two-independent-group approximation; inference is paired by question.",
        },
        "primary_metric": "mean full-sequence next-token negative log-likelihood (NLL)",
        "bootstrap": {
            "replicates": N_BOOTSTRAP,
            "seed": BOOTSTRAP_SEED,
            "confidence_interval": "percentile 95% CI resampling selected questions/items",
        },
        "long_sequence_policy": "No truncation. Sequences exceeding the usable context are scored by sliding windows; each next-token target is counted once.",
    },
    "model_results": {},
}

for model_path in _models:
    try:
        all_results["model_results"][model_path] = evaluate_model(model_path, examples, _batch_size)
    except Exception as exc:
        print(f"\nERROR evaluating {model_path}: {exc}")
        raise  # Avoid inference on an incomplete or non-aligned model set.

print("\n" + "=" * 80)
print("PAIRED QUESTION-LEVEL STATISTICAL COMPARISONS")
print("=" * 80)

pairwise_stats = compute_pairwise_statistics(
    all_results["model_results"],
    n_bootstrap=N_BOOTSTRAP,
    seed=BOOTSTRAP_SEED,
)
all_results["pairwise_statistics"] = pairwise_stats

for comp_name, comp_data in pairwise_stats.items():
    print(f"\n{comp_name}:")
    print(f"  Mean loss difference (model 1 - model 2): {comp_data['mean_loss_diff']:+.4f}")
    print(f"  Paired t = {comp_data['t_statistic']:.3f}")
    print(f"  Raw p = {comp_data['p_value_raw']:.6g}; Holm-adjusted p = {comp_data['p_value_holm']:.6g}")
    print(f"  Paired Cohen's dz = {comp_data['cohens_dz_paired']:.3f} ({comp_data['effect_size']})")
    print(f"  95% paired bootstrap CI for difference: [{comp_data['bootstrap_ci_95'][0]:.4f}, {comp_data['bootstrap_ci_95'][1]:.4f}]")
    print(f"  Lower-loss model: {comp_data['winner_lower_loss']} ({comp_data['advantage_percent']:.1f}% relative difference)")
    print("  Significant after Holm at 0.05: " + ("yes" if comp_data["significant_after_holm_05"] else "no"))

print("\n" + "=" * 80)
print("SUMMARY (lower full-sequence loss is better on this benchmark)")
print("=" * 80)
models_sorted = sorted(
    all_results["model_results"].items(),
    key=lambda item: item[1]["mean_loss"],
)
for rank, (model_path, model_results) in enumerate(models_sorted, 1):
    print(f"{rank}. {model_path.split('/')[-1]}")
    print(
        f"   Mean full-sequence loss: {model_results['mean_loss']:.4f} "
        f"(95% bootstrap CI: [{model_results['ci_95_lower']:.4f}, "
        f"{model_results['ci_95_upper']:.4f}])"
    )

print(f"\nSaving results to: {_output}")
with open(_output, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

if _visualize:
    print("\nGenerating visualizations...")
    generate_visualizations(all_results, Path(_output).parent)

print("\n" + "=" * 80)
print("COMPLETE")
print("=" * 80)
print(f"Evaluated {len(all_results['model_results'])} models on {len(examples)} unique questions")
print(f"Results saved to {_output}")


SOCIAL REASONING EVALUATION

Power-analysis reference:
  Effect size parameter: d=0.20 (small standardized effect)
  Nominal power: 80%; two-sided alpha: 0.05
  Fixed target sample: 393 unique questions
  Model comparisons are paired by question; the original sample-size formula is retained as the planning reference.

Loading reduced Social Reasoning dataset from most_preferred.jsonl...
  Available unique questions: 955
  Randomly selected: 393 (seed=42, no replacement)

Total evaluated items: 393
Unique question IDs: 393

Evaluating: Qwen3-8B
Loading model...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Computing full-sequence losses: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 393/393 [00:59<00:00,  6.56it/s]



Results for Qwen3-8B:
  Items: 393
  Mean full-sequence loss: 2.4739
  SD across items: 0.6274
  95% bootstrap CI: [2.4118, 2.5358]
  Long examples scored with sliding windows: 0 (max tokenized length=553, context limit=40960)

Evaluating: Qwen3-8B-Base
Loading model...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

...

Computing full-sequence losses: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 393/393 [00:55<00:00,  7.13it/s]



Results for llama-3.1-8b-base:
  Items: 393
  Mean full-sequence loss: 2.2212
  SD across items: 0.5538
  95% bootstrap CI: [2.1663, 2.2755]
  Long examples scored with sliding windows: 0 (max tokenized length=554, context limit=131072)

Evaluating: llama-3.1-8B-instruct
Loading model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Computing full-sequence losses: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 393/393 [00:56<00:00,  7.01it/s]



Results for llama-3.1-8B-instruct:
  Items: 393
  Mean full-sequence loss: 2.3284
  SD across items: 0.5785
  95% bootstrap CI: [2.2711, 2.3856]
  Long examples scored with sliding windows: 0 (max tokenized length=554, context limit=131072)

PAIRED QUESTION-LEVEL STATISTICAL COMPARISONS

Qwen3-8B_vs_Qwen3-8B-Base:
  Mean loss difference (model 1 - model 2): +0.2937
  Paired t = 51.162
  Raw p = 1.35556e-175; Holm-adjusted p = 8.13334e-175
  Paired Cohen's dz = 2.581 (large)
  95% paired bootstrap CI for difference: [0.2825, 0.3051]
  Lower-loss model: Qwen3-8B-Base (11.9% relative difference)
  Significant after Holm at 0.05: yes

Qwen3-8B_vs_llama-3.1-8b-base:
  Mean loss difference (model 1 - model 2): +0.2528
  Paired t = 33.481
  Raw p = 5.08244e-117; Holm-adjusted p = 2.54122e-116
  Paired Cohen's dz = 1.689 (large)
  95% paired bootstrap CI for difference: [0.2379, 0.2678]
  Lower-loss model: llama-3.1-8b-base (10.2% relative difference)
  Significant after Holm at 0.05: yes

Qw

In [10]:
all_results

{'metadata': {'dataset_file': 'most_preferred.jsonl',
  'n_examples_total': 393,
  'sampling_method': 'simple_random_sampling_without_replacement',
  'sampling_seed': 42,
  'power_analysis': {'target_power': 0.8,
   'effect_size_d': 0.2,
   'alpha_two_sided': 0.05,
   'required_n': 393,
   'note': 'N is retained from the original two-independent-group approximation; inference is paired by question.'},
  'primary_metric': 'mean full-sequence next-token negative log-likelihood (NLL)',
  'bootstrap': {'replicates': 10000,
   'seed': 2026,
   'confidence_interval': 'percentile 95% CI resampling selected questions/items'},
  'long_sequence_policy': 'No truncation. Sequences exceeding the usable context are scored by sliding windows; each next-token target is counted once.'},
 'model_results': {'Qwen3-8B': {'question_ids': ['...'],
   'losses': [2.1998026371002197,
    1.1714742183685303,
    2.886259078979492,
    3.3192741870880127,
    2.916138172149658,
    1.5569508075714111,
    1.4423